In [ ]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
from imblearn.over_sampling import SMOTE
from matplotlib.colors import ListedColormap


import seaborn as sns
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1. Data Loading and Exploration

In [ ]:
data = load_wine()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name='target')

In [ ]:
X

In [ ]:
y

In [ ]:
print("Shape of X:", X.shape)
print("Shape of y:", y.shape)
print("Classes distribution:\n", y.value_counts())

# 2. Exploratory Data Analysis (EDA)

In [ ]:
# Scatter plots
sns.pairplot(pd.concat([X, y], axis=1), hue='target')
plt.show()

In [ ]:

# Correlation heatmap
plt.figure(figsize=(12,8))
sns.heatmap(X.corr(), annot=True, cmap='coolwarm')
plt.show()

# 3. Feature Engineering

In [ ]:
X["magnesium per alcohol"] = X["magnesium"] / X["alcohol"]

# 4. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 5. Feature Scaling

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 6. Handling Class Imbalance

In [ ]:
# Check if imbalance exists
print("Training class distribution:\n", y_train.value_counts())

In [ ]:
# Apply SMOTE to balance classes
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)
print("Balanced training class distribution:\n", pd.Series(y_train_res).value_counts())

# 7. Hyperparameter Tuning with GridSearchCV

In [ ]:
param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan', 'minkowski', 'cosine'],
    'p': [1, 2, 3]
}

knn = KNeighborsClassifier()
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(knn, param_grid, cv=cv, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train_res, y_train_res)

print("Best parameters found:", grid_search.best_params_)
print("Best cross-validated accuracy:", grid_search.best_score_)

# 7.6. Train the best model

In [ ]:
best_knn = grid_search.best_estimator_
best_knn.fit(X_train_res, y_train_res)

# 7.7. Decision Boundary

As mlxtend nuked exisiting libraries upon attempted installation, so it has not been run on my computer

In [ ]:
from mlxtend.plotting import plot_decision_regions

# Select first two features for visualization
X_train_2d = X_train_res[:, :2]
X_test_2d = X_test_scaled[:, :2]  # scale test set to same first two features

# Train KNN on these two features
best_knn_2d = KNeighborsClassifier(
    n_neighbors=grid_search.best_params_['n_neighbors'],
    weights=grid_search.best_params_['weights'],
    metric=grid_search.best_params_['metric'],
    p=grid_search.best_params_['p']
)
best_knn_2d.fit(X_train_2d, y_train_res)

# Plot decision boundary
plt.figure(figsize=(10,6))
plot_decision_regions(X_train_2d, np.array(y_train_res), clf=best_knn_2d, legend=2)

plt.xlabel(data.feature_names[0])
plt.ylabel(data.feature_names[1])
plt.title("KNN Decision Boundary (mlxtend) with first 2 features")
plt.show()

# MODEL EVALUATION

# 1. Predict on the test set

In [ ]:
y_pred = best_knn.predict(X_test_scaled)


# 2. Model Evaluation

In [ ]:
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)

# 3. Classification Report

In [ ]:
cr = classification_report(y_test, y_pred)
print("Classification Report:\n", cr)

In [ ]:
print("Test Precision:", precision_score(y_test, y_pred, average='weighted'))
print("Test Recall:", recall_score(y_test, y_pred, average='weighted'))
print("Test F1-Score:", f1_score(y_test, y_pred, average='weighted'))
print("Test Accuracy:", accuracy_score(y_test, y_pred))